# 03 Transforms Semantic STFT And Correctness

`OctoSense.transforms` is OctoSense's `Semantic-Aware Operators`
layer. It solves the problem that wireless sensing preprocessing is
often built from ad hoc scripts, so semantic mistakes can slip in
when an operator receives the wrong input state or silently changes
the physical meaning of the signal. The design uses composable
operators with explicit semantic updates, so preprocessing stays
reusable, verifiable, and ready for later model adaptation. This
notebook shows that result on one real Widar3 sample by walking
through `STFT -> Magnitude -> Normalize -> AutoAlign`, checking
each intermediate output, and showing how the sample moves from raw
CSI semantics toward model-ready features.

### Set Up Transform And Visualization Utilities

Import the public transform operators, visualization helpers, and
timing tools that support a detailed, correctness-first walkthrough
of one real sample.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = (NOTEBOOK_DIR / "../..").resolve()
SRC_ROOT = REPO_ROOT / "src"
TEST_DATA_ROOT = NOTEBOOK_DIR / "data"
PRESET_ROOT = NOTEBOOK_DIR / "presets"
NOTEBOOK_TREE_DEPTH = 2
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from loguru import logger
import octosense as octo
import numpy as np
import time
import torch
import matplotlib.pyplot as plt
from octosense.datasets.storage.downloader import download_dataset
from octosense.transforms.adapters.auto_align import AutoAlign
from octosense.transforms.ops.complex import Magnitude
from octosense.transforms.ops.normalize import Normalize
from octosense.transforms.ops.spectral import STFT
from octosense.viz import plot_csi_amplitude_over_time


logger.remove()
logger.add(sys.stderr, colorize=False, format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}")
logger.info("Notebook setup completed with repository-relative paths")
logger.info("Default describe tree render depth set to {}", NOTEBOOK_TREE_DEPTH)

### Build And Validate The Semantic Transform Chain Step By Step

Load one real Widar3 sample, construct the canonical composed
transform chain, inspect every intermediate representation, and
verify that the composed call matches the explicit stepwise
application.

In [ ]:
logger.info("Reading one real Widar3 CSI sample for the semantic transform walkthrough")
dataset_root = TEST_DATA_ROOT / "widar3"
canonical_sample_path = (
    dataset_root
    / "CSI"
    / "20181211"
    / "user3"
    / "user3-1-1-1-1-r1.dat"
)
if canonical_sample_path.exists():
    logger.info("Using existing notebook-local Widar3 sample root at {}", dataset_root.relative_to(REPO_ROOT))
else:
    logger.info("Downloading Widar3 sample payload into {}", dataset_root.relative_to(REPO_ROOT))
    dataset_root = download_dataset("widar3", root=dataset_root, part="sample", force=False)
    logger.info("Using downloaded Widar3 sample root at {}", dataset_root.relative_to(REPO_ROOT))

load_started = time.perf_counter()
dataset_view = octo.datasets.load(
    "widar3",
    modalities=["wifi"],
    variant="sample",
    split_scheme="sample",
    task_binding="gesture",
    path=str(dataset_root),
)
load_elapsed = time.perf_counter() - load_started
test_view = dataset_view.get_split("test")
raw_sample, raw_label = test_view[0]
reader_channel = int(dataset_view.dataset_metadata.extra["reader_channel"])
assert reader_channel == 165

rfnet_handle = octo.models.load("rfnet", entry_overrides={"num_classes": 2})
transform_chain = octo.transforms.compose(
    [
        STFT(axis_name="time", n_fft=64, hop_length=32, onesided=False),
        Magnitude(),
        Normalize(method="zscore", per_sample=True),
        AutoAlign(
            rfnet_handle.get_input_contract(),
            axis_map={"time": "frame", "feature": ("freq", "subc", "tx", "rx")},
        ),
    ]
)
print("Canonical transform declaration:")
for index, transform in enumerate(transform_chain.transforms, start=1):
    print(f"{index}. {transform.__class__.__name__}")

stft_only, magnitude_op, normalize_op, align_op = transform_chain.transforms
stft_sample = stft_only(raw_sample)
magnitude_sample = magnitude_op(stft_sample)
normalized_sample = normalize_op(magnitude_sample)
aligned_sample = align_op(normalized_sample)
composed_aligned_sample = transform_chain(raw_sample)
print(raw_sample.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
print(raw_sample.metadata.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
print(stft_sample.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
print(normalized_sample.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
print(aligned_sample.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
logger.info(
    "STFT propagates semantics from axes {} to {} and exposes coord keys {}",
    raw_sample.axis_schema.axes,
    stft_sample.axis_schema.axes,
    tuple(sorted(stft_sample.metadata.coords)),
)
sample_rate_hz = float(raw_sample.metadata.sample_rate)
stft_freq_bins = int(stft_sample.shape[stft_sample.axis_schema.axes.index("freq")])
stft_bin_resolution_hz = sample_rate_hz / float(stft_only.n_fft)
stft_max_positive_freq_hz = sample_rate_hz / 2.0 - stft_bin_resolution_hz
logger.info(
    "STFT uses {} freq bins and spans [{:.3f}, {:.3f}] Hz with {:.3f} Hz resolution",
    stft_freq_bins,
    -sample_rate_hz / 2.0,
    stft_max_positive_freq_hz,
    stft_bin_resolution_hz,
)
logger.info("Loaded Widar3 dataset view in {:.6f}s", load_elapsed)
logger.info("Widar3 dataset-config reader channel: {}", reader_channel)
logger.info("Selected raw label from test split: {}", raw_label)
logger.info(
    "Sample metadata scalars: reader_id={}, center_freq={}, bandwidth={}, sample_rate={}",
    raw_sample.metadata.reader_id,
    raw_sample.metadata.center_freq,
    raw_sample.metadata.bandwidth,
    raw_sample.metadata.sample_rate,
)
logger.info("Tree rendering can be compressed with render(depth=NOTEBOOK_TREE_DEPTH)")
print(transform_chain.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
logger.info(
    "Composed semantic transform chain matches explicit stepwise application: {}",
    torch.allclose(
        composed_aligned_sample.as_tensor(),
        aligned_sample.as_tensor(),
        atol=1e-6,
        rtol=1e-5,
    ),
)
logger.info("Composed semantic transform chain finished for one already-read sample")

### Visualize How The Representation Changes Across The Transform Stages

Plot the raw signal together with the metadata-backed STFT
spectrogram. The visualization stops before `AutoAlign` because the
aligned tensor no longer carries physical CSI plot semantics.

In [ ]:
logger.info(
    "Semantic transform chain operators: {}",
    [transform.__class__.__name__ for transform in transform_chain.transforms],
)

aligned_axes = aligned_sample.axis_schema.axes
assert tuple(raw_sample.axis_schema.axes) == ("time", "subc", "tx", "rx")
assert tuple(aligned_axes) == ("time", "feature")
assert set(("freq", "frame", "subc", "tx", "rx")).issubset(set(stft_sample.axis_schema.axes))

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
plot_csi_amplitude_over_time(raw_sample, tx_index=0, rx_index=0, ax=axes[0], title="Input sample\nCSI amplitude over time")

subc_coord = raw_sample.metadata.get_coord("subc")
if subc_coord is not None and subc_coord.values is not None:
    subc_values = np.asarray(subc_coord.values)
else:
    subc_values = np.arange(raw_sample.shape[raw_sample.axis_schema.axes.index("subc")])
if np.issubdtype(subc_values.dtype, np.number) and np.any(subc_values < 0) and np.any(subc_values > 0):
    selected_subc_index = int(np.argmin(np.abs(subc_values.astype(np.float64))))
else:
    selected_subc_index = int(len(subc_values) // 2)
selected_subc_value = subc_values[selected_subc_index]

freq_values = np.asarray(stft_sample.metadata.get_coord("freq").values, dtype=np.float64)
frame_values = np.asarray(stft_sample.metadata.get_coord("frame").values, dtype=np.float64)
spectrogram = stft_sample.as_tensor().abs()[selected_subc_index, 0, 0].detach().cpu().numpy()
positive_freq_mask = freq_values >= 0.0
freq_values = freq_values[positive_freq_mask]
spectrogram = spectrogram[positive_freq_mask, :]

if frame_values.size == 1:
    frame_edges = np.array([frame_values[0] - 0.5, frame_values[0] + 0.5], dtype=np.float64)
else:
    frame_step = np.diff(frame_values)
    frame_midpoints = frame_values[:-1] + frame_step / 2.0
    frame_edges = np.concatenate(
        ([frame_values[0] - frame_step[0] / 2.0], frame_midpoints, [frame_values[-1] + frame_step[-1] / 2.0])
    )

if freq_values.size == 1:
    freq_edges = np.array([freq_values[0] - 0.5, freq_values[0] + 0.5], dtype=np.float64)
else:
    freq_step = np.diff(freq_values)
    freq_midpoints = freq_values[:-1] + freq_step / 2.0
    freq_edges = np.concatenate(
        ([freq_values[0] - freq_step[0] / 2.0], freq_midpoints, [freq_values[-1] + freq_step[-1] / 2.0])
    )

mesh = axes[1].pcolormesh(frame_edges, freq_edges, spectrogram, shading="auto", cmap="magma")
axes[1].set_title(f"After STFT\nreal freq axis at subc={selected_subc_value}")
axes[1].set_xlabel("Frame (s)")
axes[1].set_ylabel("Freq (Hz)")
fig.colorbar(mesh, ax=axes[1])
fig.tight_layout()
plt.show()